# [통신 실습 1] AM – 진폭 변조와 포락선 복조

**AM(Amplitude Modulation, 진폭 변조)** 은 보내려는 메시지 신호의 크기에 따라 **반송파의 진폭**을 바꾸는 방식입니다. AM 라디오 방송이 대표적인 예입니다.

$$s(t) = [1 + k_a\,m(t)]\cos(2\pi f_c t)$$

- $m(t)$: 메시지 신호, $f_c$: 반송파 주파수, $k_a$: 변조 지수
- 변조 지수 $k_a \le 1$ 이면 포락선(겉모양 곡선)이 원래 메시지 모양을 그대로 따라가므로, **포락선 검파**만으로 복조할 수 있습니다.
- 복조 = 정류(절댓값) → 저역통과필터(이동 평균) → 직류 성분 제거

> 위에서부터 차례로 `Shift + Enter`로 실행하세요.

## 0. 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False   # 그래프의 마이너스(-) 기호 깨짐 방지
print("한글 폰트 설정 완료")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. 시간 축 및 파라미터 설정

In [ ]:
# 1. 시간 축 및 파라미터 설정
fs = 100000  # 샘플링 주파수 (100kHz)
t = np.linspace(0, 0.01, int(fs * 0.01))  # 10ms 동안 관측

fm = 100     # 메시지 신호 주파수 (100Hz)
fc = 2000    # 반송파 주파수 (2kHz)
ka = 0.8     # 변조 지수 (Modulation Index, 1 이하일 때 정상 복조)

## 2. 신호 생성

In [ ]:
# 2. 신호 생성
m_t = np.cos(2 * np.pi * fm * t)                  # 원본 메시지 신호
carrier = np.cos(2 * np.pi * fc * t)              # 반송파 신호
s_t = (1 + ka * m_t) * carrier                   # AM 변조 신호

## 3. 포락선 검파 (Envelope Detection - 단순 절대값 및 절반파 정류)

In [ ]:
# 3. 포락선 검파 (Envelope Detection - 단순 절대값 및 절반파 정류)
rectified = np.abs(s_t)
# 이동 평균 필터(Low-pass Filter 역할)를 이용한 포락선 추출
window_size = int(fs / fc)
demod_signal = np.convolve(rectified, np.ones(window_size)/window_size, mode='same')
demod_signal = (demod_signal * np.pi / 2 - 1) / ka   # 정류 평균(2/π) 보정 → DC(1) 제거 → 스케일링  # DC 성분 제거 및 스케일링

## 4. 시각화

In [ ]:
# 4. 시각화
plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
plt.plot(t * 1000, m_t, 'b', label="메시지 신호 m(t)")
plt.title("1. 원본 메시지 신호 (100Hz)")
plt.xlabel("시간 (ms)")
plt.ylabel("진폭")
plt.grid(True)
plt.legend()

plt.subplot(3, 1, 2)
plt.plot(t * 1000, s_t, 'g', label="AM 변조 신호 s(t)")
plt.plot(t * 1000, (1 + ka * m_t), 'r--', alpha=0.6, label="포락선 (Upper Envelope)")
plt.title(f"2. AM 변조 신호 (반송파: {fc}Hz, 변조지수: {ka})")
plt.xlabel("시간 (ms)")
plt.ylabel("진폭")
plt.grid(True)
plt.legend()

plt.subplot(3, 1, 3)
plt.plot(t * 1000, m_t, 'b--', label="원본 메시지", alpha=0.5)
plt.plot(t * 1000, demod_signal, 'r', label="복조된 신호")
plt.title("3. 포락선 검파를 통한 복조 결과")
plt.xlabel("시간 (ms)")
plt.ylabel("진폭")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

### 원본 코드에서 수정한 부분
- 정류한 코사인파의 평균은 원래 진폭의 2/π 배이므로, π/2를 곱해 보정해야 복조 신호의 크기가 원본과 같아집니다.

## 직접 해 보기
- 변조 지수 `ka`를 1.5로 바꾸면 포락선이 어떻게 되는지(과변조) 관찰해 보세요.
- 반송파 `fc`를 500Hz로 낮추면 복조 결과가 어떻게 달라지나요?
- 값을 바꾼 뒤에는 바뀐 셀부터 아래로 다시 실행하세요.